# Kaggle finetune + ONNX export for SkyTNT/midi-model

End-to-end recipe for the diploma project. Runs on a single T4 GPU.

## Inputs you must attach to this notebook
1. **This repo** uploaded as a Kaggle Dataset (or `git clone`'d in cell 2). Folder name doesn't matter — we cd into it.
2. **Your raw MIDI** as a Kaggle Dataset (e.g. `your-username/my-midi-corpus`). Files can be nested.

## Outputs (under `/kaggle/working/artifacts/`)
- `onnx/model_base.onnx`
- `onnx/model_token.onnx`
- `tokenizer/tokenizer_config.json`
- `examples/sample_*.mid` (smoke test)

Download the artifacts at the end of the run, drop them under `artifacts/` in your local repo, and the JUCE plugin will pick them up automatically.

## 1. Environment setup
Enable GPU + Internet under "Settings" before running.

In [ ]:
import os, shutil, sys
from pathlib import Path

# Pick the dataset path Kaggle gave us.
REPO_INPUT = "/kaggle/input/midi-generation-plugin"   # your code dataset
MIDI_INPUT = "/kaggle/input/my-midi-corpus"           # your MIDI dataset

assert os.path.exists(REPO_INPUT), (
    "Attach the repo as a Kaggle dataset and set REPO_INPUT. "
    f"Not found: {REPO_INPUT}"
)

# Copy repo dataset into working directory
WORKROOT = Path("/kaggle/working/repo")
if WORKROOT.exists():
    shutil.rmtree(WORKROOT)
WORKROOT.mkdir(parents=True, exist_ok=True)

# Kaggle datasets are folders; copy everything
shutil.copytree(REPO_INPUT, WORKROOT, dirs_exist_ok=True)

# Find the real project root that contains skytnt_adapter/
project_root = None
for c in [WORKROOT] + [p for p in WORKROOT.iterdir() if p.is_dir()]:
    if (c / "skytnt_adapter").is_dir():
        project_root = c
        break

if project_root is None:
    for p in WORKROOT.rglob("skytnt_adapter"):
        if p.is_dir():
            project_root = p.parent
            break

assert project_root is not None, (
    "Could not find project root containing skytnt_adapter/. "
    f"Top-level under {WORKROOT}: {[p.name for p in WORKROOT.iterdir() if p.is_dir()]}"
)

# Ensure both the notebook kernel and subprocesses can import from this folder
os.environ["PYTHONPATH"] = str(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

%cd {str(project_root)}
print("Project root:", project_root)
print("CWD:", os.getcwd())
print("Has skytnt_adapter:", os.path.isdir("skytnt_adapter"))

# Quick import smoke test in-kernel and in a subprocess
import skytnt_adapter
print("Import OK (kernel):", skytnt_adapter.__file__)
!python -c "import os, skytnt_adapter; print('Import OK (subprocess):', os.getcwd(), skytnt_adapter.__file__)"

!ls

In [ ]:
!pip install -q -r requirements.txt

## 2. Sanity-check the dataset
Counts how many MIDI files we found.

In [ ]:
import os

n = 0
for root, _, files in os.walk(MIDI_INPUT):
    for f in files:
        if f.lower().endswith((".mid", ".midi")):
            n += 1
print(f"Found {n} MIDI files under {MIDI_INPUT}")
assert n > 0, "No MIDI files found - check MIDI_INPUT."

## 3. Download SkyTNT pretrained weights
(Skip if you have your own pretraining.)

In [ ]:
from huggingface_hub import snapshot_download

PRETRAINED_REPO = "skytnt/midi-model-tv2o-medium"
pretrained_dir = snapshot_download(repo_id=PRETRAINED_REPO, allow_patterns=["*.safetensors", "*.bin", "*.ckpt", "config.json"])
print("pretrained at:", pretrained_dir)
!ls -la "$pretrained_dir"

## 4. Finetune
T4 has 16 GB VRAM, so use bf16 mixed precision + small batch + grad accumulation.

**If you see `filtered out … [3000, 384000]` and only ~100–200 train files while the folder has thousands of MIDIs:** the default max file size (375 KB) cuts almost everything. The cell below sets `--max-file-bytes` much higher so most of the corpus is used.

**Overfitting** with tiny train is normal. More real train files + lower LR + higher weight decay + `--sample-seq` (less VRAM on long pieces) helps. For very small data, try `--task lora` instead of full fine-tune.

In [ ]:
%env PYTHONUNBUFFERED=1
!python -m skytnt_adapter.finetune_skytnt \
    --data "$MIDI_INPUT" \
    --pretrained "$pretrained_dir" \
    --output checkpoints/skytnt \
    --config tv2o-medium \
    --max-len 2048 \
    --min-file-bytes 512 --max-file-bytes 8000000 \
    --batch-size 1 --batch-size-val 1 \
    --workers 2 --workers-val 2 \
    --acc-grad 4 \
    --max-step 4000 --warmup-step 200 --val-step 100 \
    --val-fraction 0.15 --early-stop-patience 8 \
    --lr 5e-5 --weight-decay 0.05 \
    --sample-seq \
    --precision bf16-mixed \
    --accelerator gpu --devices 1 \
    --ckpt-weights-only --no-save-last

## 4b. Training metrics

In [ ]:
import glob, os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# Find the latest Lightning CSV log
log_dirs = sorted(glob.glob("checkpoints/skytnt/lightning_logs/version_*/metrics.csv"))
if not log_dirs:
    log_dirs = sorted(glob.glob("checkpoints/skytnt/**/metrics.csv", recursive=True))

assert log_dirs, "metrics.csv not found — did training finish?"
csv_path = log_dirs[-1]
print(f"Reading: {csv_path}")

df = pd.read_csv(csv_path)
print(df.tail())

def smooth(s, w=5):
    return s.rolling(w, min_periods=1, center=True).mean()

def val_series(col):
    mask = df[col].notna() if col in df.columns else pd.Series(False, index=df.index)
    return df.loc[mask, "step"], df.loc[mask, col]

def pct(s):
    return s * 100

# ── Layout: 3 rows × 3 cols ──────────────────────────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(16, 12))
fig.suptitle("Fine-tune metrics  –  SkyTNT / midi-model", fontsize=15, fontweight="bold")

# Row 0 ── Loss & Perplexity
ax = axes[0, 0]
mask = df["train/loss"].notna()
if mask.any():
    s, v = df.loc[mask, "step"], df.loc[mask, "train/loss"]
    ax.plot(s, v, alpha=0.2, color="steelblue", linewidth=0.7, label="raw")
    ax.plot(s, smooth(v), color="steelblue", linewidth=2, label="smoothed")
    ax.set_title("Train loss"); ax.set_ylabel("cross-entropy")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[0, 1]
s, v = val_series("val/loss")
if len(s):
    ax.plot(s, v, color="tomato", marker="o", markersize=5, linewidth=2, label="val loss")
    ax.set_title("Validation loss"); ax.set_ylabel("cross-entropy")
    ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[0, 2]
s, v = val_series("val/perplexity")
if len(s):
    ax.plot(s, v, color="darkorange", marker="s", markersize=5, linewidth=2)
    ax.set_title("Perplexity  exp(val_loss)")
    ax.set_ylabel("perplexity"); ax.grid(alpha=0.3)

# Row 1 ── Accuracy family
ax = axes[1, 0]
s, v = val_series("val/acc")
if len(s):
    ax.plot(s, pct(v), color="seagreen", marker="o", markersize=5, linewidth=2, label="top-1")
s3, v3 = val_series("val/top3_acc")
if len(s3):
    ax.plot(s3, pct(v3), color="mediumseagreen", marker="^", markersize=5,
            linewidth=2, linestyle="--", label="top-3")
s5, v5 = val_series("val/top5_acc")
if len(s5):
    ax.plot(s5, pct(v5), color="lightgreen", marker="D", markersize=5,
            linewidth=2, linestyle=":", label="top-5")
ax.set_title("Token accuracy (top-k)"); ax.set_ylabel("accuracy (%)")
ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%.1f%%"))
ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1, 1]
s, v = val_series("val/f1_micro")
if len(s):
    ax.plot(s, v, color="royalblue", marker="o", markersize=5, linewidth=2, label="micro")
s2, v2 = val_series("val/f1_macro")
if len(s2):
    ax.plot(s2, v2, color="cornflowerblue", marker="s", markersize=5,
            linewidth=2, linestyle="--", label="macro")
ax.set_title("F1 score"); ax.set_ylabel("F1")
ax.set_ylim(0, 1); ax.legend(fontsize=8); ax.grid(alpha=0.3)

ax = axes[1, 2]
s, v = val_series("val/recall_macro")
if len(s):
    ax.plot(s, v, color="orchid", marker="o", markersize=5, linewidth=2, label="recall macro")
s2, v2 = val_series("val/prec_macro")
if len(s2):
    ax.plot(s2, v2, color="mediumpurple", marker="s", markersize=5,
            linewidth=2, linestyle="--", label="precision macro")
ax.set_title("Recall & Precision (macro)"); ax.set_ylabel("score")
ax.set_ylim(0, 1); ax.legend(fontsize=8); ax.grid(alpha=0.3)

# Row 2 ── LR + summary table
ax = axes[2, 0]
s, v = val_series("train/lr")
if not len(s):
    mask = df["train/lr"].notna()
    s, v = df.loc[mask, "step"], df.loc[mask, "train/lr"]
if len(s):
    ax.plot(s, v, color="slategray", linewidth=2)
    ax.set_title("Learning rate schedule"); ax.set_ylabel("LR")
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
    ax.grid(alpha=0.3)

# Summary table in last two cells
ax = axes[2, 1]
ax.axis("off")
metric_cols = ["val/loss", "val/perplexity", "val/acc", "val/top3_acc",
               "val/top5_acc", "val/f1_micro", "val/f1_macro",
               "val/recall_macro", "val/prec_macro"]
rows = []
for col in metric_cols:
    if col in df.columns:
        vals = df[col].dropna()
        if len(vals):
            label = col.replace("val/", "")
            rows.append([label, f"{vals.iloc[0]:.4f}", f"{vals.iloc[-1]:.4f}",
                         f"{vals.max():.4f}", f"{vals.min():.4f}"])
if rows:
    tbl = ax.table(cellText=rows,
                   colLabels=["metric", "first", "last", "best", "worst"],
                   loc="center", cellLoc="center")
    tbl.auto_set_font_size(False); tbl.set_fontsize(9)
    tbl.scale(1, 1.5)
    ax.set_title("Summary", pad=12)

ax = axes[2, 2]
ax.axis("off")

for ax_row in axes:
    for ax in ax_row:
        if ax.get_xlabel() == "":
            ax.set_xlabel("step")

plt.tight_layout()
out_png = "checkpoints/skytnt/training_metrics.png"
plt.savefig(out_png, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved → {out_png}")

## 5. Export to ONNX
Produces both `model_base.onnx` and `model_token.onnx` plus `tokenizer_config.json`.

In [ ]:
!python -m skytnt_adapter.export_skytnt_onnx \
    --ckpt checkpoints/skytnt \
    --config tv2o-medium \
    --out-dir /kaggle/working/artifacts/onnx \
    --tokenizer-out /kaggle/working/artifacts/tokenizer/tokenizer_config.json

## 6. Smoke test the exported ONNX models
Generates 2 short MIDI snippets to confirm the runtime works.

In [ ]:
!python -m skytnt_adapter.sample_generate \
    --mode onnx \
    --base /kaggle/working/artifacts/onnx/model_base.onnx \
    --token /kaggle/working/artifacts/onnx/model_token.onnx \
    --tokenizer /kaggle/working/artifacts/tokenizer/tokenizer_config.json \
    --out /kaggle/working/artifacts/examples \
    --num 2 --max-len 256

## 7. Bundle artifacts
After the cell below, download `/kaggle/working/artifacts.zip` from the Output panel and drop the contents into `artifacts/` in your local repo.
The JUCE plugin auto-discovers them at startup.

In [ ]:
!cd /kaggle/working && zip -r artifacts.zip artifacts && ls -la artifacts.zip